# 단일 에이전트에서 멀티 에이전트로

* 대부분의 에이전트는 하나로 시작
* 사용하는 툴의 수는 증가하고 에이전트엥 맡기려하는 문제의 범위가 넓어질수록 멀티 에이전트 패턴을 도입하면 전체적인 성능과 신뢰성을 높일 수 있음

## 에이전트는 몇 개나 필요할까?

* 성능을 개선할 필요가 있을 때만 복잡성을 추가하는 것이 좋음
* 적절한 에이전트의 개수, 구성 방식은 작업의 난이도, 도구의 개수, 환경의 복잡성에 따라 크게 달라짐

### 단일 에이전트 시나리오

* 단일 에이전트 시스템은 난이도가 그리 높지 않은 작업, 제한된 수의 도구, 복잡성이 낮은 환경에 적합

* 주요 장점
    * 단순성
    * 낮은 리소스 요구사항
    * 지연시간

* 단일 에이전트 설정 방법 예시

In [ ]:
from __future__ import annotations
"""
supply_chain_logistics_agent.py
재고 관리, 운송 작업, 공급업체 관계 및 창고 최적화를 처리하는
공급망 및 물류 관리 에이전트를 위한 LangGraph 워크플로
"""
import os
import json
import operator
import builtins
from typing import Annotated, Sequence, TypedDict, Optional

from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.messages.tool import ToolMessage
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

from traceloop.sdk import Traceloop
from src.common.observability.loki_logger import log_to_loki

@tool
def manage_inventory(sku: Optional[str] = None, **kwargs) -> str:
    """재고 수준, 재고 보충, 감사 및 최적화 전략을 관리"""
    print(f"[도구] manage_inventory(sku={sku}, kwargs={kwargs})")
    log_to_loki("tool.manage_inventory", f"sku={sku}")
    return "inventory_management_initiated"

@tool
def track_shipments(origin: Optional[str] = None, **kwargs) -> str:
    """배송 상태, 지연 사항을 추적하고 배송 물류를 조정"""
    print(f"[도구] track_shipments(origin={origin}, kwargs={kwargs})")
    log_to_loki("tool.track_shipments", f"origin={origin}")
    return "shipment_tracking_updated"

@tool
def evaluate_suppliers(supplier_name: Optional[str] = None, **kwargs) -> str:
    """공급업체 성과를 평가하고 감사를 수행하며 공급업체 관계를 관리"""
    print(f"[도구] evaluate_suppliers(supplier_name={supplier_name}, kwargs={kwargs})")
    log_to_loki("tool.evaluate_suppliers", f"supplier_name={supplier_name}")
    return "supplier_evaluation_complete"

@tool
def optimize_warehouse(operation_type: Optional[str] = None, **kwargs) -> str:
    """창고 운영, 레이아웃, 용량 및 보관 효율성을 최적화"""
    print(f"[도구] optimize_warehouse(operation_type={operation_type}, kwargs={kwargs})")
    log_to_loki("tool.optimize_warehouse", f"operation_type={operation_type}")
    return "warehouse_optimization_initiated"

@tool
def forecast_demand(season: Optional[str] = None, **kwargs) -> str:
    """수요 패턴, 계절적 추세를 분석하고 예측 모델을 생성"""
    print(f"[도구] forecast_demand(season={season}, kwargs={kwargs})")
    log_to_loki("tool.forecast_demand", f"season={season}")
    return "demand_forecast_generated"

@tool
def manage_quality(supplier: Optional[str] = None, **kwargs) -> str:
    """품질 관리, 결함 추적 및 공급업체 품질 표준을 관리"""
    print(f"[도구] manage_quality(supplier={supplier}, kwargs={kwargs})")
    log_to_loki("tool.manage_quality", f"supplier={supplier}")
    return "quality_management_initiated"

@tool
def arrange_shipping(shipping_type: Optional[str] = None, **kwargs) -> str:
    """배송 방법, 특급 배송 및 복합 운송을 준비"""
    print(f"[도구] arrange_shipping(shipping_type={shipping_type}, kwargs={kwargs})")
    log_to_loki("tool.arrange_shipping", f"shipping_type={shipping_type}")
    return "shipping_arranged"

@tool
def coordinate_operations(operation_type: Optional[str] = None, **kwargs) -> str:
    """크로스도킹, 통합 및 이동과 같은 복잡한 작업을 조정"""
    print(f"[도구] coordinate_operations(operation_type={operation_type}, kwargs={kwargs})")
    log_to_loki("tool.coordinate_operations", f"operation_type={operation_type}")
    return "operations_coordinated"

@tool
def manage_special_handling(product_type: Optional[str] = None, **kwargs) -> str:
    """위험물, 콜드체인 및 민감한 제품에 대한 특수 요구사항을 처리"""
    print(f"[도구] manage_special_handling(product_type={product_type}, kwargs={kwargs})")
    log_to_loki("tool.manage_special_handling", f"product_type={product_type}")
    return "special_handling_managed"

@tool
def handle_compliance(compliance_type: Optional[str] = None, **kwargs) -> str:
    """규제 준수, 세관, 문서화 및 인증을 관리"""
    print(f"[도구] handle_compliance(compliance_type={compliance_type}, kwargs={kwargs})")
    log_to_loki("tool.handle_compliance", f"compliance_type={compliance_type}")
    return "compliance_handled"

@tool
def process_returns(returned_quantity: Optional[str] = None, **kwargs) -> str:
    """반품, 역물류 및 제품 처리를 처리"""
    print(f"[도구] process_returns(returned_quantity={returned_quantity}, kwargs={kwargs})")
    log_to_loki("tool.process_returns", f"returned_quantity={returned_quantity}")
    return "returns_processed"

@tool
def scale_operations(scaling_type: Optional[str] = None, **kwargs) -> str:
    """성수기, 용량 계획 및 인력 관리를 위한 운영을 확장"""
    print(f"[도구] scale_operations(scaling_type={scaling_type}, kwargs={kwargs})")
    log_to_loki("tool.scale_operations", f"scaling_type={scaling_type}")
    return "operations_scaled"

@tool
def optimize_costs(cost_type: Optional[str] = None, **kwargs) -> str:
    """운송, 보관 및 운영 비용을 분석하고 최적화"""
    print(f"[도구] optimize_costs(cost_type={cost_type}, kwargs={kwargs})")
    log_to_loki("tool.optimize_costs", f"cost_type={cost_type}")
    return "cost_optimization_initiated"

@tool
def optimize_delivery(delivery_type: Optional[str] = None, **kwargs) -> str:
    """배송 경로, 라스트마일 물류 및 지속가능성 이니셔티브를 최적화"""
    print(f"[도구] optimize_delivery(delivery_type={delivery_type}, kwargs={kwargs})")
    log_to_loki("tool.optimize_delivery", f"delivery_type={delivery_type}")
    return "delivery_optimization_complete"

@tool
def manage_disruption(disruption_type: Optional[str] = None, **kwargs) -> str:
    """공급망 중단, 비상 계획 및 위험 완화를 관리"""
    print(f"[도구] manage_disruption(disruption_type={disruption_type}, kwargs={kwargs})")
    log_to_loki("tool.manage_disruption", f"disruption_type={disruption_type}")
    return "disruption_managed"

@tool
def send_logistics_response(operation_id: Optional[str] = None, message: Optional[str] = None) -> str:
    """이해관계자에게 물류 업데이트, 권장 사항 또는 상태 보고서를 전송"""
    print(f"[도구] send_logistics_response → {message}")
    log_to_loki("tool.send_logistics_response", f"operation_id={operation_id}, message={message}")
    return "logistics_response_sent"

TOOLS = [
    manage_inventory, track_shipments, evaluate_suppliers, optimize_warehouse,
    forecast_demand, manage_quality, arrange_shipping, coordinate_operations,
    manage_special_handling, handle_compliance, process_returns, scale_operations,
    optimize_costs, optimize_delivery, manage_disruption, send_logistics_response
]

In [ ]:
Traceloop.init(disable_batch=True, app_name="supply_chain_logistics_agent_langgraph")

llm = init_chat_model(model="gpt-5-mini", callbacks=[StreamingStdOutCallbackHandler()],  
    verbose=True).bind_tools(TOOLS)

class AgentState(TypedDict):
    operation: Optional[dict]  # 공급망 운영 정보
    messages: Annotated[Sequence[BaseMessage], operator.add]

def call_model(state: AgentState):
    history = state["messages"]
    
    # 누락되거나 불완전한 작업 데이터를 적절히 처리
    operation = state.get("operation", {})
    if not operation:
        operation = {"operation_id": "UNKNOWN", "type": "general", "priority": "medium", "status": "active"}
    
    operation_json = json.dumps(operation, ensure_ascii=False)
    system_prompt = f"""
        당신은 숙련된 공급망 및 물류 관리 전문가입니다.
        전문 분야:
        - 재고 관리 및 수요 예측
        - 운송 및 배송 최적화
        - 공급업체 관계 관리 및 평가
        - 창고 운영 및 용량 계획
        - 품질 관리 및 규정 준수 관리
        - 비용 최적화 및 운영 효율성
        - 위험 관리 및 중단 대응
        - 지속가능성 및 친환경 물류 이니셔티브

        공급망 운영을 관리할 때:
        1) 물류 과제 또는 기회를 분석합니다
        2) 적절한 공급망 관리 도구를 호출합니다
        3) send_logistics_response로 권장 사항을 제공합니다
        4) 비용, 효율성, 품질 및 지속가능성 영향을 고려합니다
        5) 고객 만족도와 비즈니스 연속성을 우선시합니다

        항상 비용 최적화와 서비스 품질 및 위험 완화의 균형을 유지하십시오.

        작업: {operation_json}"""

    full = [SystemMessage(content=system_prompt)] + history

    first: ToolMessage | BaseMessage = llm.invoke(full)
    messages = [first]

    if getattr(first, "tool_calls", None):
        for tc in first.tool_calls:
            print(first)
            print(tc['name'])
            fn = next(t for t in TOOLS if t.name == tc['name'])
            out = fn.invoke(tc["args"])
            messages.append(ToolMessage(content=str(out), tool_call_id=tc["id"]))

        second = llm.invoke(full + messages)
        messages.append(second)

    return {"messages": messages}

def construct_graph():
    g = StateGraph(AgentState)
    g.add_node("assistant", call_model)
    g.set_entry_point("assistant")
    return g.compile()

graph = construct_graph()

if __name__ == "__main__":
    Traceloop.init(disable_batch=True, app_name="supply_chain_logistics_agent_langgraph")
    example = {"operation_id": "OP-12345", "type": "inventory_management", "priority": "high", "location": "Warehouse A"}
    convo = [HumanMessage(content="SKU-12345 재고가 심각하게 부족합니다. 현재 재고는 50개이지만 미처리 주문이 200개입니다. 재주문 전략은 무엇입니까?")]
    result = graph.invoke({"operation": example, "messages": convo})
    for m in result["messages"]:
        print(f"{m.type}: {m.content}")

* 해당 구조는 에이전트 간 통신이 없어 오버헤드를 최소화해 지연시간을 낮게 유지
* 대부분의 사용 사례에서는 도구의 책임의 개수가 늘어날 때 핵심 병목이 생김  
-> 멀티 에이전트로 넘어가기 전 싱글 에이전트 프레임워크 안에서 확장하는 방식을 고려할 것

### 멀티 에이전트 시나리오

* 여러 에이전트가 공통 목표를 달성하기 위해 협력
* 작업이 복잡하고 다양한 도구 모음, 병렬 처리, 동적인 환경에 대한 적응력이 필요한 문제에 특히 유리함
* 각 에이전트에 특정 역할이나 전문 분야를 할당해 시스템이 각 에이전트의 강점을 효과적으로 활용

* 멀티 에이전트 시스템 예시
    * 16개의 도구를 세 개의 전문화된 에이전트로 분해

In [ ]:
from typing import Annotated, Sequence, TypedDict, Optional

from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage

from langchain_core.messages.tool import ToolMessage
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

from traceloop.sdk import Traceloop
from src.common.observability.loki_logger import log_to_loki

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass 

os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "http://localhost:4317"
os.environ["OTEL_EXPORTER_OTLP_INSECURE"] = "true"

# 모든 전문가를 위한 공유 도구
@tool
def send_logistics_response(operation_id: Optional[str] = None, message: Optional[str] = None) -> str:
    """이해관계자에게 물류 업데이트, 권장 사항 또는 상태 보고서를 전송합니다."""
    print(f"[도구] send_logistics_response → {message}")
    log_to_loki("tool.send_logistics_response", f"operation_id={operation_id}, message={message}")
    return "logistics_response_sent"

# 재고 및 창고 전문가 도구
@tool
def manage_inventory(sku: Optional[str] = None, **kwargs) -> str:
    """재고 수준, 재고 보충, 감사 및 최적화 전략을 관리합니다."""
    print(f"[도구] manage_inventory(sku={sku}, kwargs={kwargs})")
    log_to_loki("tool.manage_inventory", f"sku={sku}")
    return "inventory_management_initiated"

@tool
def optimize_warehouse(operation_type: Optional[str] = None, **kwargs) -> str:
    """창고 운영, 레이아웃, 용량 및 보관 효율성을 최적화합니다."""
    print(f"[도구] optimize_warehouse(operation_type={operation_type}, kwargs={kwargs})")
    log_to_loki("tool.optimize_warehouse", f"operation_type={operation_type}")
    return "warehouse_optimization_initiated"

@tool
def forecast_demand(season: Optional[str] = None, **kwargs) -> str:
    """수요 패턴, 계절적 추세를 분석하고 예측 모델을 생성합니다."""
    print(f"[도구] forecast_demand(season={season}, kwargs={kwargs})")
    log_to_loki("tool.forecast_demand", f"season={season}")
    return "demand_forecast_generated"

@tool
def manage_quality(supplier: Optional[str] = None, **kwargs) -> str:
    """품질 관리, 결함 추적 및 공급업체 품질 표준을 관리합니다."""
    print(f"[도구] manage_quality(supplier={supplier}, kwargs={kwargs})")
    log_to_loki("tool.manage_quality", f"supplier={supplier}")
    return "quality_management_initiated"

@tool
def scale_operations(scaling_type: Optional[str] = None, **kwargs) -> str:
    """성수기, 용량 계획 및 인력 관리를 위한 운영을 확장합니다."""
    print(f"[도구] scale_operations(scaling_type={scaling_type}, kwargs={kwargs})")
    log_to_loki("tool.scale_operations", f"scaling_type={scaling_type}")
    return "operations_scaled"

@tool
def optimize_costs(cost_type: Optional[str] = None, **kwargs) -> str:
    """운송, 보관 및 운영 비용을 분석하고 최적화합니다."""
    print(f"[도구] optimize_costs(cost_type={cost_type}, kwargs={kwargs})")
    log_to_loki("tool.optimize_costs", f"cost_type={cost_type}")
    return "cost_optimization_initiated"

INVENTORY_TOOLS = [manage_inventory, optimize_warehouse, forecast_demand, manage_quality, scale_operations, optimize_costs, send_logistics_response]

# 운송 및 물류 전문가 도구
@tool
def track_shipments(origin: Optional[str] = None, **kwargs) -> str:
    """배송 상태, 지연 사항을 추적하고 배송 물류를 조정합니다."""
    print(f"[도구] track_shipments(origin={origin}, kwargs={kwargs})")
    log_to_loki("tool.track_shipments", f"origin={origin}")
    return "shipment_tracking_updated"

@tool
def arrange_shipping(shipping_type: Optional[str] = None, **kwargs) -> str:
    """배송 방법, 특급 배송 및 복합 운송을 준비합니다."""
    print(f"[도구] arrange_shipping(shipping_type={shipping_type}, kwargs={kwargs})")
    log_to_loki("tool.arrange_shipping", f"shipping_type={shipping_type}")
    return "shipping_arranged"

@tool
def coordinate_operations(operation_type: Optional[str] = None, **kwargs) -> str:
    """크로스도킹, 통합 및 이동과 같은 복잡한 작업을 조정합니다."""
    print(f"[도구] coordinate_operations(operation_type={operation_type}, kwargs={kwargs})")
    log_to_loki("tool.coordinate_operations", f"operation_type={operation_type}")
    return "operations_coordinated"

@tool
def manage_special_handling(product_type: Optional[str] = None, **kwargs) -> str:
    """위험물, 콜드체인 및 민감한 제품에 대한 특수 요구사항을 처리합니다."""
    print(f"[도구] manage_special_handling(product_type={product_type}, kwargs={kwargs})")
    log_to_loki("tool.manage_special_handling", f"product_type={product_type}")
    return "special_handling_managed"

@tool
def process_returns(returned_quantity: Optional[str] = None, **kwargs) -> str:
    """반품, 역물류 및 제품 처리를 처리합니다."""
    print(f"[도구] process_returns(returned_quantity={returned_quantity}, kwargs={kwargs})")
    log_to_loki("tool.process_returns", f"returned_quantity={returned_quantity}")
    return "returns_processed"

@tool
def optimize_delivery(delivery_type: Optional[str] = None, **kwargs) -> str:
    """배송 경로, 라스트마일 물류 및 지속가능성 이니셔티브를 최적화합니다."""
    print(f"[도구] optimize_delivery(delivery_type={delivery_type}, kwargs={kwargs})")
    log_to_loki("tool.optimize_delivery", f"delivery_type={delivery_type}")
    return "delivery_optimization_complete"

@tool
def manage_disruption(disruption_type: Optional[str] = None, **kwargs) -> str:
    """공급망 중단, 비상 계획 및 위험 완화를 관리합니다."""
    print(f"[도구] manage_disruption(disruption_type={disruption_type}, kwargs={kwargs})")
    log_to_loki("tool.manage_disruption", f"disruption_type={disruption_type}")
    return "disruption_managed"

TRANSPORTATION_TOOLS = [track_shipments, arrange_shipping, coordinate_operations, manage_special_handling, process_returns, optimize_delivery, manage_disruption, send_logistics_response]

# 공급업체 및 규정 준수 전문가 도구
@tool
def evaluate_suppliers(supplier_name: Optional[str] = None, **kwargs) -> str:
    """공급업체 성과를 평가하고 감사를 수행하며 공급업체 관계를 관리합니다."""
    print(f"[도구] evaluate_suppliers(supplier_name={supplier_name}, kwargs={kwargs})")
    log_to_loki("tool.evaluate_suppliers", f"supplier_name={supplier_name}")
    return "supplier_evaluation_complete"

@tool
def handle_compliance(compliance_type: Optional[str] = None, **kwargs) -> str:
    """규제 준수, 세관, 문서화 및 인증을 관리합니다."""
    print(f"[도구] handle_compliance(compliance_type={compliance_type}, kwargs={kwargs})")
    log_to_loki("tool.handle_compliance", f"compliance_type={compliance_type}")
    return "compliance_handled"

SUPPLIER_TOOLS = [evaluate_suppliers, handle_compliance, send_logistics_response]

Traceloop.init(disable_batch=True, app_name="supply_chain_logistics_agent")
llm = init_chat_model(model="gpt-5-mini", callbacks=[StreamingStdOutCallbackHandler()], verbose=True)

# 전문화된 LLM에 도구 바인딩
inventory_llm = llm.bind_tools(INVENTORY_TOOLS)
transportation_llm = llm.bind_tools(TRANSPORTATION_TOOLS)
supplier_llm = llm.bind_tools(SUPPLIER_TOOLS)

In [ ]:
class AgentState(TypedDict):
    operation: Optional[dict]  # 공급망 운영 정보
    messages: Annotated[Sequence[BaseMessage], operator.add]

# 슈퍼바이저(관리자) 노드: 적절한 전문가에게 라우팅
def supervisor_node(state: AgentState):
    history = state["messages"]
    operation = state.get("operation", {})
    operation_json = json.dumps(operation, ensure_ascii=False)
    
    supervisor_prompt = f"""
        당신은 공급망 전문가 팀을 조정하는 슈퍼바이저입니다.
        팀원:
        - inventory: 재고 수준, 예측, 품질, 창고 최적화, 확장 및 비용을 처리합니다.
        - transportation: 배송 추적, 준비, 운영 조정, 특수 처리, 반품, 배송 최적화 및 중단을 처리합니다.
        - supplier: 공급업체 평가 및 규정 준수를 처리합니다.

        사용자 쿼리를 기반으로 처리할 팀원 한 명을 선택하세요.
        선택한 팀원의 이름(inventory, transportation 또는 supplier)만 출력하고 다른 것은 출력하지 마세요.

        작업: {operation_json}
    """

    full = [SystemMessage(content=supervisor_prompt)] + history
    response = llm.invoke(full)
    return {"messages": [response]}

# 전문가 노드 템플릿
def specialist_node(state: AgentState, specialist_llm, system_prompt: str):
    history = state["messages"]
    operation = state.get("operation", {})
    if not operation:
        operation = {"operation_id": "UNKNOWN", "type": "general", "priority": "medium", "status": "active"}
    operation_json = json.dumps(operation, ensure_ascii=False)
    full_prompt = system_prompt + f"\n\n작업: {operation_json}"
    
    full = [SystemMessage(content=full_prompt)] + history

    first: ToolMessage | BaseMessage = specialist_llm.invoke(full)
    messages = [first]

    if getattr(first, "tool_calls", None):
        for tc in first.tool_calls:
            print(first)
            print(tc['name'])
            # 도구 찾기 (모든 도구 이름이 고유하다고 가정)
            all_tools = INVENTORY_TOOLS + TRANSPORTATION_TOOLS + SUPPLIER_TOOLS
            fn = next(t for t in all_tools if t.name == tc['name'])
            out = fn.invoke(tc["args"])
            messages.append(ToolMessage(content=str(out), tool_call_id=tc["id"]))

        second = specialist_llm.invoke(full + messages)
        messages.append(second)

    return {"messages": messages}

# 재고 전문가 노드
def inventory_node(state: AgentState):
    inventory_prompt = """
        당신은 재고 및 창고 관리 전문가입니다.
        관리할 때:
          1) 재고/창고 과제를 분석합니다
          2) 적절한 도구를 호출합니다
          3) send_logistics_response로 후속 조치합니다
        비용, 효율성 및 확장성을 고려하세요.
    """
    return specialist_node(state, inventory_llm, inventory_prompt)

# 운송 전문가 노드
def transportation_node(state: AgentState):
    transportation_prompt = """
        당신은 운송 및 물류 전문가입니다.
        관리할 때:
          1) 배송/전달 과제를 분석합니다
          2) 적절한 도구를 호출합니다
          3) send_logistics_response로 후속 조치합니다
        효율성, 지속가능성 및 위험 완화를 고려하세요.
    """
    return specialist_node(state, transportation_llm, transportation_prompt)

# 공급업체 전문가 노드
def supplier_node(state: AgentState):
    supplier_prompt = """
        당신은 공급업체 관계 및 규정 준수 전문가입니다.
        관리할 때:
          1) 공급업체/규정 준수 문제를 분석합니다
          2) 적절한 도구를 호출합니다
          3) send_logistics_response로 후속 조치합니다
        성과, 규정 및 관계를 고려하세요.
    """
    return specialist_node(state, supplier_llm, supplier_prompt)


In [ ]:
# 조건부 엣지를 위한 라우팅 함수
def route_to_specialist(state: AgentState):
    last_message = state["messages"][-1]
    agent_name = last_message.content.strip().lower()
    if agent_name == "inventory":
        return "inventory"
    elif agent_name == "transportation":
        return "transportation"
    elif agent_name == "supplier":
        return "supplier"
    else:
        # 일치하는 항목이 없으면 폴백
        return END

def construct_graph():
    g = StateGraph(AgentState)
    g.add_node("supervisor", supervisor_node)
    g.add_node("inventory", inventory_node)
    g.add_node("transportation", transportation_node)
    g.add_node("supplier", supplier_node)
    
    g.set_entry_point("supervisor")
    g.add_conditional_edges("supervisor", route_to_specialist, {"inventory": "inventory", "transportation": "transportation", "supplier": "supplier"})
    
    g.add_edge("inventory", END)
    g.add_edge("transportation", END)
    g.add_edge("supplier", END)
    
    return g.compile()

graph = construct_graph()

if __name__ == "__main__":
    Traceloop.init(disable_batch=True, app_name="supply_chain_logistics_agent_langgraph")
    example = {"operation_id": "OP-12345", "type": "inventory_management", "priority": "high", "location": "Warehouse A"}
    convo = [HumanMessage(content="SKU-12345 재고가 심각하게 부족합니다. 현재 재고는 50개이지만 미처리 주문이 200개입니다. 재주문 전략은 무엇입니까?")]
    result = graph.invoke({"operation": example, "messages": convo})
    for m in result["messages"]:
        print(f"{m.type}: {m.content}")

* 적응성은 멀티 에이전트 시스템의 핵심 장점
* 다만 도전 과제도 있음
    * 상호작용하면서 조율의 복잡성이 커지고 에이전트가 조화롭게 작동하도록 정교한 통신, 동기화 메커니즘이 필요함
    * 통신 오버헤드
    * 충돌 해결과 리소스 할당을 위한 프로토콜

### 스웜

* 스웜 시스템은 많은 수의 단순한 에이전트가 개별적으로는 지능이 거의 없으나 국소적인 상호작용과 단순한 규칙을 통해 집단적으로 지능적인 창발적 행동을 만들어냄

* 스웜 시스템은 탈중앙화와 자기 조직화를 강조
* 각 에이전트는 자신만의 로컬 정책이나 행동 규칙을 따름
* 작은 업데이트를 브로드캐스트, 이웃에 반응, 공유된 신호에 기반해 적응하는 반복적인 국소 상호작용을 함

* 스웜 기반 시스템의 장점
    * 확장성
    * 견고성
    * 유연성
    * 분산 문제 해결

* 스웜은 중앙 집중식 제어가 바람직하지 않는 환경에서 효과적
* 시스템을 설계할 때는 다음의 측면에서 고유한 과제가 따름
    * 예측 가능성
    * 관측 가능성
    * 효율성

## 에이전트 추가 원칙

* 에이전트 기반 설계와 기능을 최적화하는 가이드라인
    * 작업 분해
    * 전문화
    * 파시모니
    * 조율
    * 견고성
    * 효율성

## 멀티 에이전트 조율

### 민주적 조율

* 시스템 내 모든 에이전트가 각자 동등한 의사결정 권한을 가지며 행동과 솔루션에 대해 합의에 도달하는 것을 목표로 함
* 탈중앙화된 제어가 특징
* 에이전트들은 정보를 동등하게 공유하고 협업하며 각자의 고유한 관점을 기여해 집단적으로 결정을 내림

* 장점
    * 견고성
        * 시스템에는 단일 장애 지점이 없음
    * 유연성
        * 에이전트들이 개방적으로 협업하면 집단 입력을 갱신하는 방식으로 환경 변화에 빠르게 적응할 수 있음  
        -> 민첩한 대응이 중요한 동적인 환경에서 필수적

* 민주적 조율의 고유한 과제
    * 통신 오버헤드
    * 민주적 조율 프로토콜

### 관리자 중심 조율

* 중앙집중적인 접근 방식으로 하나 이상의 에이전트르 관리자로 지정해 하위 에이전트의 행동을 감독하고 지시하는 책임을 맡김
* 관리자는 의사 결정을 내리고 작업을 분배하며 자신이 관리하는 에이전트들 사이의 충돌을 해결하는 감독 역할

* 주요 장점
    * 의사결정 과정이 간소화
    * 통신 경로가 단순해지고 조율 복잡성이 줄어듦

* 취약성
    * 단일 장애 지점이 존재
    * 확장성의 문제
    * 적응성의 문제

### 계층적 조율

* 구조화된 계층을 통해 중앙집중식 제어와 탈중앙화 제어 요소를 결합하는 다단계 조직 방식을 취함
* 상위 수준 에이전트
    * 하위 수준 에이전트를 감독하고 지시
    * 하위 에이전트에 일정 수준의 자율성을 부여

-> 완전 중앙집중식 모델보다 훨씬 많은 수의 에이전트를 더 효율적으로 관리할 수 있음

* 고유한 과제
    * 계층형 시스템 설계의 복잡성
    * 통신 지연과 반응 속도

### 액터-크리틱 접근법

* 평가 주도 반복을 가볍게 적용한 형태
* 액터
    * 답변, 계획, 행동과 같은 후보 출력을 생성하는 역할
* 크리틱
    * 사전에 정의된 평가 기준에 따라 출력을 받아들이거나 거부하는 품질 게이트 역할

* 프로세스
    * 액터가 출력을 원하는 품질 임계값에 도달할 때까지 계속해서 후보를 만들어 내고 크리틱이 이를 판정  
    -> 신뢰성과 성능을 높이기 위해 추가 추론 사이클을 사용하는 테스트 시점 연산의 한 형태

* 효과적인 상황
    * 명확한 평가 기준이나 체크리스트가 있을 때
    * 더 높은 품질이 주는 이득에 비해 추가 출력을 생성하는 비용이 수용 가능할 때
    * 한 번의 시도로는 성능이 떨어지나 재랭킹이나 필터링을 거친 접근 방식이 더 잘 작동하는 모호하거나 생성적인 작업일 때

* 액터-크리틱 접근법 예시

In [ ]:
# AgentState 정의 - candidates와 iteration 필드 추가
class AgentState(TypedDict):
    operation: Optional[dict]  # 공급망 운영 정보
    messages: Annotated[Sequence[BaseMessage], operator.add]
    candidates: Optional[list]  # Actor가 생성한 후보 계획들
    iteration: Optional[int]  # 반복 횟수

# Actor 노드: 후보 계획 생성
def actor_node(state: AgentState):
    """3개의 후보 공급망 계획을 생성"""
    history = state["messages"]
    actor_prompt = '''3개의 공급망 후보 계획을 JSON 리스트 형식으로 생성하세요.
    형식: [{'plan': '계획 설명', 'tools': [{'tool': '도구명', 'args': {...}}]}]
    각 계획은 실행 가능한 구체적인 단계와 필요한 도구를 포함해야 합니다.'''
    response = llm.invoke([SystemMessage(content=actor_prompt)] + history)
    try:
        candidates = json.loads(response.content)
    except json.JSONDecodeError:
        # JSON 파싱 실패 시 기본 후보 제공
        candidates = [{"plan": "기본 계획", "tools": []}]
    return {"candidates": candidates, "messages": state["messages"]}

# Critic 노드: 평가 및 선택/반복
def critic_node(state: AgentState):
    """후보 계획들을 평가하고 최적의 계획을 선택하거나 재생성을 요청."""
    candidates = state.get("candidates", [])
    history = state["messages"]
    
    critic_prompt = f'''다음 후보 계획들을 평가하세요: {candidates}
    
    실행 가능성(feasibility), 비용(cost), 위험도(risk) 기준으로 각각 1-10점으로 채점하세요.
    
    응답 형식 (JSON):
    {{
        "evaluations": [
            {{"plan_index": 0, "feasibility": 점수, "cost": 점수, "risk": 점수, "total": 총점}},
            ...
        ],
        "best_score": 최고점수,
        "selected": 선택된_계획_객체,
        "feedback": "개선을 위한 피드백 (점수가 8점 이하인 경우)"
    }}
    
    최고 점수가 8점 이상이면 해당 계획을 선택하고, 그렇지 않으면 재생성을 요청하세요.'''
    
    response = llm.invoke([SystemMessage(content=critic_prompt)] + history)
    
    try:
        evaluation = json.loads(response.content)
    except json.JSONDecodeError:
        # JSON 파싱 실패 시 첫 번째 후보 선택
        evaluation = {
            "best_score": 9,
            "selected": candidates[0] if candidates else {"plan": "기본 계획", "tools": []},
            "feedback": ""
        }
    
    if evaluation.get('best_score', 0) > 8:
        winning_plan = evaluation['selected']
        # 선택된 계획의 도구들을 실행
        messages = []
        for tool_info in winning_plan.get('tools', []):
            tool_name = tool_info.get('tool', '')
            tool_args = tool_info.get('args', {})
            tc = {'name': tool_name, 'args': tool_args, 'id': f'tool_{len(messages)}'}
            
            # 도구 찾기 및 실행
            try:
                fn = next(t for t in ALL_TOOLS if t.name == tool_name)
                out = fn.invoke(tool_args)
                messages.append(ToolMessage(content=str(out), tool_call_id=tc["id"]))
            except StopIteration:
                print(f"[경고] 도구를 찾을 수 없음: {tool_name}")
            except Exception as e:
                print(f"[오류] 도구 실행 실패: {tool_name}, {e}")
        
        # 최종 응답 전송
        send_logistics_response.invoke({"message": winning_plan.get('plan', '계획 실행 완료')})
        
        final_message = AIMessage(
            content=f"선택된 계획: {winning_plan.get('plan', '')} (점수: {evaluation.get('best_score', 0)})"
        )
        return {"messages": history + messages + [final_message]}
    else:
        # 반복: Actor에게 피드백 제공
        feedback_message = AIMessage(
            content=f"재생성 필요: 개선 사항 - {evaluation.get('feedback', '더 나은 계획이 필요합니다.')}"
        )
        return {"messages": history + [feedback_message]}

# Actor-Critic 그래프 구성
def construct_actor_critic_graph():
    """Actor-Critic 패턴을 사용한 공급망 관리 그래프를 구성"""
    g = StateGraph(AgentState)
    g.add_node("actor", actor_node)
    g.add_node("critic", critic_node)
    
    g.set_entry_point("actor")
    g.add_edge("actor", "critic")
    
    # 승인되지 않은 경우 다시 Actor로 돌아감 (조건부)
    def should_continue(state: AgentState) -> str:
        """Critic이 재생성을 요청했는지 확인합니다."""
        if not state.get("messages"):
            return END
        last_message = state["messages"][-1]
        if hasattr(last_message, 'content') and "재생성" in last_message.content:
            return "actor"
        return END
    
    g.add_conditional_edges("critic", should_continue)
    
    return g.compile()

* 생성보다 평가가 더 쉬울 때 특히 유용함

## 에이전틱 시스템의 자동 설계

* 에이전틱 시스템 자동 설계(ADAS)는 수작업으로 만든 아키텍처에서 벗어나 스스로를 설계하고 평가하며 반복적으로 개선할 수 있는 시스템으로 나아가는 에이전트 개발의 변혁적 접근

* 핵심 아이디어
    * 상위 수준의 메타 에이전트 탐색(MAS) 알고리즘이 에이전틱 시스템을 자동으로 생성하고 평가하고 개선하도록

-> 사람의 직접 개입 없이도 복잡하고 변화무쌍한 환경에 적응하고 자신의 역량을 지속적으로 향상시킬 수 있게 만들 잠재력이 존재

* ADAS에서 파운데이션 모델은 에이전트 아키텍처 내에서 유연한 범용 모듈 역할을 함
* 완전히 새로운 구조와 모듈을 스스로 발명할 수 있게 함으로써 기존 접근을 넘어서는 것을 목표로 함

* 프레임워크의 핵심 구성 요소
    * 탐색 공간
        * 표현 가능한 에이전틱 아키텍처의 범위를 정의
    * 탐색 알고리즘
        * 이 공간 내에서의 탐색 전략을 결정
    * 평가 함수
        * 성능, 견고성, 효율성과 같은 목표에 비춰 후보 에이전트의 효과성을 정량화

* ADAS의 백본은 코드를 통해 에이전트를 정의한다는 개념
* ADAS의 힘은 코드 기반 접근에 있음  
-> 시간에 따라 재정의, 수정, 최적화될 수 있는 유연한 구성물로 다루기 때문

* MAS
    * 메타 에이전트가 어떻게 에이전트 시스템을 자율적으로 생성하고 개선할 수 있는지를 보여주는 구체적 방법
    * 메타 에이전트
        * 설계자로서 행동
        * 새로운 에이전트를 정의하는 코드를 작성
        * 다양한 작업에 대해 테스트
    * 반복적인 사이클로 작동

* ADAS 연구적 접근 방식 예시

In [ ]:
class LLMAgentBase:
    """
    LLM 에이전트의 기본 클래스, 다양한 출력 형식을 위해 설정 가능합니다.
    """
    def __init__(self, output_fields: list, agent_name: str,
                 role='helpful assistant', model='gpt-5-mini', temperature=0.5) -> None:
        self.output_fields = output_fields
        self.agent_name = agent_name
        self.role = role
        self.model = model
        self.temperature = temperature
        self.id = random_id()  # Assume random_id from utils

    def generate_prompt(self, input_infos, instruction, output_description) -> tuple:
        output_fields_and_description = {key: output_description.get(key, f"Your {key}.") for key in self.output_fields}
        system_prompt = ROLE_DESC(self.role) + "\n\n" + FORMAT_INST(output_fields_and_description)

        input_infos_text = ''
        for input_info in input_infos:
            if isinstance(input_info, Info):
                (field_name, author, content, iteration_idx) = input_info
                if author == self.__repr__():
                    author += ' (yourself)'
                if field_name == 'task':
                    input_infos_text += f'# Your Task:\n{content}\n\n'
                elif iteration_idx != -1:
                    input_infos_text += f'### {field_name} #{iteration_idx + 1} by {author}:\n{content}\n\n'
                else:
                    input_infos_text += f'### {field_name} by {author}:\n{content}\n\n'

        prompt = input_infos_text + instruction
        return system_prompt, prompt

    def query(self, input_infos: list, instruction, output_description, iteration_idx=-1) -> list:
        system_prompt, prompt = self.generate_prompt(input_infos, instruction, output_description)
        try:
            response_json = get_json_response_from_gpt(prompt, self.model, system_prompt, self.temperature)
            assert len(response_json) == len(self.output_fields), "not returning enough fields"
        except Exception as e:
            response_json = {key: '' for key in self.output_fields if key not in response_json}
            for key in list(response_json):
                if key not in self.output_fields:
                    del response_json[key]
        output_infos = [Info(key, self.__repr__(), value, iteration_idx) for key, value in response_json.items()]
        return output_infos

In [ ]:
def search(args, task):
    archive = task.get_init_archive() # 또는 기존 아카이브 불러오기
    for n in range(args.n_generation):
        # 아카이브에서 프롬프트 생성
        msg_list = [{"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt}]

        next_solution = get_json_response_from_gpt_reflect(msg_list, args.model)
        # 초기 생성
        # Reflexion: 두 단계로 개선
        next_solution = reflect_and_refine(msg_list, task.get_reflexion_prompt())
        # 리플렉시온을 위한 의사 코드
        # 평가 및 디버깅
        acc_list = evaluate_forward_fn(args, next_solution["code"], task)
        next_solution['fitness'] = bootstrap_confidence_interval(acc_list)
        archive.append(next_solution)

def evaluate_forward_fn(args, forward_str, task):
    # 에이전트 코드를 동적으로 함수로 로드
    exec(forward_str, globals(), namespace)
    func = namespace['forward'] # 단일 함수가 있다고 가정
    data = task.load_data(SEARCHING_MODE) # 검증 또는 테스트 데이터
    task_queue = task.prepare_task_queue(data)
    # 병렬 평가
    with ThreadPoolExecutor() as executor:
        acc_list = list(executor.map(process_item, task_queue))
        # process_item: func 실행 후 정답과 비교하여 점수 계산
        return acc_list

* 에이전트들은 새로운 도메인과 모델에 적용되더라도 높은 성능을 유지하는 경향이 있음
* 윤리적 측면과 기술적 측면 모두에 대한 신중한 고려가 필요함

## 에이전트 통신 기법

### 로컬 통신과 분산 통신

* 단일 디바이스나 단일 프로세스 같은 소규모 환경에서는 에이전트가 직접 함수 호출, 공유 메모리 또는 메모리 내 메시지 큐를 통해 통신하는 경우가 많음  
    -> 확장성의 한계가 있음

* 로컬 배포 환경
    * 메모리 내 라우터를 사용해 에이전트 간 메시지 전달과 도구 호출을 조율  
        -> 단일 스레드나 단일 에이전트 설정이 주를 이루는 단계에서는 유용

### A2A 프로토콜

* 자율 에이전트들이 협력하여 더 복잡한 목표를 달성할 수 있도록 돕는 시도
* 에이전트가 내부 로직이나 구현 세부 사항을 노출하지 않고도 서로 탐색, 협업 조율, 구조화된 요청을 교환할 수 있는 표준화된 크로스 플랫폼 메커니즘을 제공

* A2A는 이기종 에이전트가 HTTP 기반 전송 계층 위에서 상호 운용될 수 있게 함으로써 보편적인 멀티 에이전트 협업 언어를 형성할 잠재력을 가짐

* A2A의 핵심은 에이전트 카드
    * 각 에이전트가 자신의 신원, 기능, 엔드포인트, 지원 인증 방식을 알리기 위해 발행하는 기계 판독 가능한 JSON 기술서
    * 상대 에이전트를 탐색, 평가, 보안 통신 채널을 협상할 수 있음

* 에이전트 카드 예시

In [ ]:
# 에이전트 카드 (발견을 위한 JSON 설명자)
agent_card = {
    "name": "SummarizerAgent",
    "description": "텍스트 요약을 수행하는 AI 에이전트입니다.",
    "protocolVersion": "1.0",
    "url": "http://localhost:8000",
    "provider": {
        "organization": "Example Org",
        "url": "https://example.org"
    },
    "capabilities": {
        "streaming": False,
        "pushNotifications": False,
        "stateTransitionHistory": False
    },
    "skills": [
        {
            "id": "summarize-text",
            "name": "텍스트 요약",
            "description": "주어진 텍스트를 간결하게 요약합니다.",
            "tags": ["summarization", "nlp", "text-processing"],
            "examples": [
                "이 기사를 요약해 주세요",
                "다음 내용을 간단히 정리해 주세요"
            ]
        }
    ],
    "defaultInputModes": ["text/plain"],
    "defaultOutputModes": ["text/plain"],
    "security": []  # 프로덕션에서는 OAuth2, API 키 등을 설정하세요
}

In [ ]:
import requests
import json

# 단계 1: 에이전트 검색 - A2A 스펙 준수 well-known URI
card_url = 'http://localhost:8000/.well-known/agent-card.json'
response = requests.get(card_url)
if response.status_code != 200:
    print("에이전트 카드 가져오기 실패")
    exit()

agent_card = response.json()
print("발견된 에이전트 카드:", json.dumps(agent_card, indent=2, ensure_ascii=False))

# 단계 2: 핸드셰이크 (버전 및 기능 확인)
if agent_card.get('protocolVersion', '').split('.')[0] != '1':
    print("호환되지 않는 프로토콜 버전")
    exit()

# skills 확인
skills = agent_card.get('skills', [])
skill_ids = [s.get('id') for s in skills]
if "summarize-text" not in skill_ids:
    print("필요한 스킬이 지원되지 않음")
    exit()
print("핸드셰이크 성공: 에이전트가 호환됩니다.")

# 단계 3: A2A 스펙 준수 JSON-RPC 요청 (message/send)
rpc_url = agent_card['url']  # 에이전트 기본 URL로 POST
rpc_request = {
    "jsonrpc": "2.0",
    "method": "message/send",
    "params": {
        "contextId": str(uuid.uuid4()),
        "message": {
            "role": "user",
            "parts": [
                {
                    "text": "이것은 요약이 필요한 긴 예제 텍스트입니다. 멀티 에이전트 시스템, 통신 프로토콜, A2A와 같은 표준을 사용하여 에이전트들이 자율적으로 협업하는 방법을 논의합니다."
                }
            ]
        }
    },
    "id": 123  # 고유한 요청 ID
}

response = requests.post(rpc_url, json=rpc_request)
if response.status_code == 200:
    rpc_response = response.json()
    print("\nRPC 응답:", json.dumps(rpc_response, indent=2, ensure_ascii=False))
    
    # 결과 파싱
    if 'result' in rpc_response:
        result = rpc_response['result']
        print(f"\n태스크 ID: {result.get('id')}")
        print(f"상태: {result.get('status', {}).get('state')}")
        
        artifacts = result.get('artifacts', [])
        if artifacts:
            for artifact in artifacts:
                for part in artifact.get('parts', []):
                    if 'text' in part:
                        print(f"\n요약 결과:\n{part['text']}")
else:
    print("오류:", response.status_code, response.text)

In [ ]:
# agent_server.py

# 요청 처리
import os
from openai import OpenAI

class AgentHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        # A2A 스펙: /.well-known/agent-card.json (섹션 8.2, 14.3)
        if self.path == '/.well-known/agent-card.json':
            self.send_response(200)
            self.send_header('Content-type', 'application/json; charset=utf-8')
            self.end_headers()
            self.wfile.write(json.dumps(agent_card, ensure_ascii=False).encode('utf-8'))
        else:
            self.send_response(404)
            self.end_headers()

    def do_POST(self):
        if self.path == '/':
            content_length = int(self.headers['Content-Length'])
            post_data = self.rfile.read(content_length)
            rpc_request = json.loads(post_data)
            
            # A2A JSON-RPC 요청 처리 (섹션 9.4.1 message/send)
            if rpc_request.get('jsonrpc') == '2.0' and rpc_request['method'] == 'message/send':
                params = rpc_request.get('params', {})
                message = params.get('message', {})
                parts = message.get('parts', [])
                
                # 텍스트 파트 추출 (섹션 4.1.6 Part)
                text = ""
                for part in parts:
                    if 'text' in part:
                        text += part['text']
                
                # OpenAI API를 사용한 실제 LLM 요약
                client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
                try:
                    llm_response = client.chat.completions.create(
                        model="gpt-4o-mini",
                        messages=[
                            {"role": "system", "content": "당신은 간결한 요약을 제공하는 유용한 어시스턴트입니다."},
                            {"role": "user", "content": f"다음 텍스트를 요약하세요:\n{text}"}
                        ],
                    )
                    summary = llm_response.choices[0].message.content.strip()
                except Exception as e:
                    summary = f"요약 중 오류 발생: {str(e)}"
                
                # A2A 스펙 준수 응답 (섹션 4.1.1 Task, 4.1.2 TaskStatus)
                task_id = str(uuid.uuid4())
                response = {
                    "jsonrpc": "2.0",
                    "result": {
                        "id": task_id,
                        "contextId": params.get('contextId', str(uuid.uuid4())),
                        "status": {
                            "state": "completed"
                        },
                        "artifacts": [
                            {
                                "parts": [{"text": summary}]
                            }
                        ]
                    },
                    "id": rpc_request['id']
                }
                self.send_response(200)
                self.send_header('Content-type', 'application/json; charset=utf-8')
                self.end_headers()
                self.wfile.write(json.dumps(response, ensure_ascii=False).encode('utf-8'))
            else:
                # JSON-RPC 오류 처리 (섹션 9.5)
                error_response = {
                    "jsonrpc": "2.0",
                    "error": {"code": -32601, "message": "Method not found"},
                    "id": rpc_request.get('id')
                }
                self.send_response(200)  # JSON-RPC 에러도 HTTP 200으로 반환
                self.send_header('Content-type', 'application/json; charset=utf-8')
                self.end_headers()
                self.wfile.write(json.dumps(error_response, ensure_ascii=False).encode('utf-8'))
        else:
            self.send_response(404)
            self.end_headers()


if __name__ == '__main__':
    server_address = ('', 8000)
    httpd = HTTPServer(server_address, AgentHandler)
    print("A2A 에이전트 서버를 시작합니다. 주소: http://localhost:8000")
    print("Agent Card: http://localhost:8000/.well-known/agent-card.json")
    httpd.serve_forever()

## 메시지 브로커와 이벤트 버스

* point-to-point 통신 방식은 에이전트 기반 시스템의 규모가 확장됨에 따라 불안정해지고 유연성이 떨어짐  
    -> 일반적인 대안은 메시지 브로커나 이벤트 버스를 도입
* 송신자와 수신자의 결합을 분리하고 에이전트들이 공유 통신 패브릭을 통해 비동기적으로 상호 작용할 수 있게 함  
    -> 느슨히 결합된 멀티 에이전트 아키텍처에서 확장 가능하고 장애 허용성이 있으며 관측 가능한 워크플로를 확립해 줌

* 메시지 브로커를 통합하는 사례
    * 수퍼바이저는 고유 토픽에 작업을 발행, 전문가는 이를 비동기적으로 구독해 관련 메시지만 처리  
        -> 에이전트가 분리되므로 독립적인 확장이 가능해지고 내결함성이 확보되며 그래프를 다시 작성하지 않고도 새 에이전트를 쉽게 추가할 수 있음
    * 아파치 카프카
    * 레디스 스트림과 래빗MQ
    * NATS(Neural Autonomic Transport System)

* 레디스 스트림을 사용한 예시

In [ ]:
import redis
import json
import uuid

# Redis용 메시지 직렬화 헬퍼
def serialize_messages(messages: Sequence[BaseMessage]) -> list[dict]:
    return [m.model_dump() for m in messages]

# 슈퍼바이저: 전문가를 결정하고 Redis 스트림에 작업을 게시
def supervisor_publish(operation: dict, messages: Sequence[BaseMessage]) -> str:
    operation_json = json.dumps(operation, ensure_ascii=False)
    
    supervisor_prompt = (
        "당신은 공급망 전문가 팀을 조율하는 슈퍼바이저입니다.\n"
        "팀 구성원:\n"
        "- inventory: 재고 수준, 예측, 품질, 창고 최적화, 확장 및 비용을 처리합니다.\n"
        "- transportation: 배송 추적, 준비, 운영 조정, 특수 처리, 반품, 배달 최적화 및 중단을 처리합니다.\n"
        "- supplier: 공급업체 평가 및 규정 준수를 처리합니다.\n"
        "\n"
        "사용자 쿼리에 따라 처리할 팀 구성원 한 명을 선택하세요.\n"
        "선택된 구성원의 이름(inventory, transportation 또는 supplier)만 출력하고, 다른 것은 출력하지 마세요.\n\n"
        f"작업: {operation_json}"
    )

    full = [SystemMessage(content=supervisor_prompt)] + messages
    response = llm.invoke(full)
    
    agent_name = response.content.strip().lower()
    
    r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT)
    task_id = str(uuid.uuid4())
    task_message = {
        'task_id': task_id,
        'agent': agent_name,
        'operation': operation,
        'messages': serialize_messages(messages)
    }
    r.xadd(TASK_STREAM, {'data': json.dumps(task_message)})
    
    return task_id

In [ ]:
# 재고 전문가 소비자 루프
def inventory_consumer():
    r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT)
    last_id = '0'
    print(f"[재고 전문가] 시작됨, Redis 연결: {r.ping()}")
    inventory_prompt = (
        "당신은 재고 및 창고 관리 전문가입니다.\n"
        "관리할 때:\n"
        "  1) 재고/창고 문제를 분석합니다\n"
        "  2) 필요한 도구를 호출합니다 (예: manage_inventory, optimize_costs 등)\n"
        "  3) 작업 완료 후 send_logistics_response를 호출하여 최종 응답을 전달합니다\n"
        "  4) send_logistics_response 호출 후에는 더 이상 도구를 호출하지 않습니다\n"
        "비용, 효율성 및 확장성을 고려하세요.\n"
        "최종 응답에서는 구체적인 권장 사항과 실행 계획을 제공하세요."
    )
    
    while True:
        messages = r.xread({TASK_STREAM: last_id}, count=1, block=5000)
        if messages:
            stream, entries = messages[0]
            for entry_id, entry_data in entries:
                task = json.loads(entry_data[b'data'])
                print(f"[재고 전문가] 작업 수신: task_id={task['task_id']}, agent={task['agent']}")
                if task['agent'] == 'inventory':
                    print(f"[재고 전문가] 작업 처리 시작: {task['task_id']}")
                    state = {
                        'operation': task['operation'],
                        'messages': deserialize_messages(task['messages'])
                    }
                    result = specialist_node(state, inventory_llm, inventory_prompt)
                    response_message = {
                        'task_id': task['task_id'],
                        'from': 'inventory',
                        'result': {
                            'messages': serialize_messages(result['messages'])
                        }
                    }
                    r.xadd(RESPONSE_STREAM, {'data': json.dumps(response_message)})
                    print(f"[재고 전문가] 응답 게시됨: {task['task_id']}")
                last_id = entry_id


In [ ]:
# task_id로 응답 대기 함수
def wait_for_response(task_id: str, timeout: int = 60) -> dict:
    r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT)
    last_id = '0'
    start_time = time.time()
    
    print(f"[대기 중] task_id={task_id}에 대한 응답을 기다립니다...")
    
    while time.time() - start_time < timeout:
        # count를 늘려서 더 많은 메시지를 한 번에 읽습니다
        messages = r.xread({RESPONSE_STREAM: last_id}, count=10, block=2000)
        if messages:
            stream, entries = messages[0]
            for entry_id, entry_data in entries:
                response = json.loads(entry_data[b'data'])
                print(f"[확인] 응답 수신: task_id={response['task_id']}, from={response['from']}")
                if response['task_id'] == task_id:
                    print(f"[성공] 일치하는 응답을 찾았습니다!")
                    return response
                # 다음 읽기를 위해 last_id 업데이트
                last_id = entry_id
    
    print(f"[타임아웃] {timeout}초 내에 응답을 받지 못했습니다")
    raise TimeoutError("제한 시간 내에 응답을 받지 못했습니다")

* 메시지 버스는 에이전트 간 느슨한 결합을 지원  
    -> 확장성을 유연하게 확보하고 로깅 파이프라인을 통한 관측 가능성을 높이며 놓친 메시지를 재생할 수 있음

## 액터 프레임워크

* 액터 프레임워크는 메시징과 연산을 하나의 통합된 모델로 제공
* 액터
    * 단순히 메시지를 교환하는 것에 그치지 않고 자체 상태와 기능을 캡슐화
    * 순차 처리를 보장 -> 경쟁 상태나 공유 상태로 인한 버그를 원천 제거

* 정교한 분산 처리, 복원력, 동적 확장이 필요한 상황에서 진가를 발휘
    * 코드를 수정하지 않고도 액터를 마이그레이션하거나 복제할 수 있는 위치 투명성을 제공
    * 장애 발생 시 자동 복구를 위한 내장된 감독 기능으로 운영 오버헤드가 적음

* 클러스터 구축, 액터 라이프사이클 모니터링 같은 인프라 투자는 에이전트 수가 일정 수준을 넘어서거나 가변적인 워크로드를 처리할 때 빛을 발함

* 대표적인 프레임워크
    * 레이
    * 올리언스
    * 아카

* 액터 스타일 설계는 각 에이전트가 고유한 정체성, 역할, 내부 상태를 유지하는 멀티 에이전트 조율과 자연스럽게 맞닿음
* 공유 상태나 전역 제어 대신 메시지 전달로 에이전트를 동적 호출, 이벤트에 반응, 복잡한 워크플로 관리 가능

* 레이 액터 예시

In [ ]:
# 전문가를 위한 Ray 액터 (세션별 격리)
# 주의: LLM 객체는 직렬화가 불가능하므로 액터 내부에서 생성해야 함
@ray.remote
class SpecialistActor:
    def __init__(self, name: str, tools_key: str, system_prompt: str):
        self.name = name
        # 액터 내부에서 LLM 초기화 (직렬화 문제 회피)
        base_llm = init_chat_model(model="gpt-5-mini", verbose=True)
        tools = TOOLS_MAP[tools_key]
        self.llm = base_llm.bind_tools(tools)
        self.tools = {t.name: t for t in tools}
        self.prompt = system_prompt
        self.internal_state = {}  # 세션별 격리된 상태, 예: 세션 내 추적용

    def process_task(self, operation: dict, messages: Sequence[BaseMessage]):
        if not operation:
            operation = {"operation_id": "알 수 없음", "type": "일반", "priority": "중간", "status": "활성"}
        operation_json = json.dumps(operation, ensure_ascii=False)
        full_prompt = self.prompt + f"\n\n작업: {operation_json}"
        
        full = [SystemMessage(content=full_prompt)] + messages

        first = self.llm.invoke(full)
        result_messages = [first]

        if hasattr(first, "tool_calls"):
            for tc in first.tool_calls:
                print(first)
                print(tc['name'])
                fn = self.tools.get(tc['name'])
                if fn:
                    out = fn.invoke(tc["args"])
                    result_messages.append(ToolMessage(content=str(out), tool_call_id=tc["id"]))

            second = self.llm.invoke(full + result_messages)
            result_messages.append(second)

        # 내부 상태 업데이트 (예: 세션 내에서 처리된 단계 추적)
        step_key = str(len(self.internal_state) + 1)  # 또는 더 구체적인 키 사용
        self.internal_state[step_key] = {"status": "처리됨", "timestamp": time.time()}

        return {"messages": result_messages}

    def get_state(self):
        return self.internal_state  # 전체 세션 상태 반환

In [ ]:
# 세션 관리자 액터: 세션별 전문가 액터를 추적
@ray.remote
class SessionManager:
    def __init__(self):
        self.sessions: Dict[str, Dict[str, ray.actor.ActorHandle]] = {}  # session_id -> {agent_name: actor}

    def get_or_create_actor(self, session_id: str, agent_name: str, prompt: str):
        if session_id not in self.sessions:
            self.sessions[session_id] = {}
        if agent_name not in self.sessions[session_id]:
            # LLM은 액터 내부에서 생성됨 (직렬화 불가능하므로)
            actor = SpecialistActor.remote(agent_name, agent_name, prompt)
            self.sessions[session_id][agent_name] = actor
        return self.sessions[session_id][agent_name]

    def get_session_state(self, session_id: str, agent_name: str):
        if session_id in self.sessions and agent_name in self.sessions[session_id]:
            actor = self.sessions[session_id][agent_name]
            return actor.get_state.remote()  # future 반환
        return None


* 비동기 워크플로를 처리하고 영속적 메모리를 유지하며 분산 인프라에 깔끔하게 통합할 수 있음

## 오케스트레이션 및 워크플로 엔진

* 워크플로 오케스트레이션 도구는 더 높은 수준의 추상화 도구를 제공해 복잡한 에이전틱 시스템에서 지속성, 복구 가능성을 보장
* 실패 가능성, 장시간 실행이 프로세스에 포함될 때 특히 유용

* 템포럴은 장기 실행 작업, 재시도, 실패 복구 기능을 갖춘 지속적이고 상태 유지형 워크플로를 제공
    * 멀티 에이전트 시스템을 관리하는 데 적합
    * 여러 서비스나 에이전트에 걸쳐 장시간 수행되는 비즈니스 로직을 깔끔하게 캡슐화하는 추상화를 제공

* 템포럴을 활용한 멀티 에이전트 오케스트레이션 예시

In [ ]:
from datetime import timedelta
from temporalio import workflow
from temporalio.common import RetryPolicy

# 액티비티들은 다른 곳에서 정의되어 있다고 가정
# 각 액티비티는 operation 딕셔너리와 messages 리스트를 입력으로 받아 결과를 반환
@workflow.defn
class SupplyChainWorkflow:
    @workflow.run
    async def run(self, operation: dict, initial_messages: list) -> dict:
        # 1단계: 재시도 정책이 적용된 재고 관리
        inventory_result = await workflow.execute_activity(
            "inventory_activity",
            {"operation": operation, "messages": initial_messages},
            start_to_close_timeout=timedelta(seconds=30),
            retry_policy=RetryPolicy(maximum_attempts=3)
        )

        # 상태를 갱신한 뒤 운송 단계로 직행
        updated_messages = initial_messages + inventory_result["messages"]
        transportation_result = await workflow.execute_activity(
            "transportation_activity",
            {"operation": operation, "messages": updated_messages},
            start_to_close_timeout=timedelta(seconds=30),
            retry_policy=RetryPolicy(maximum_attempts=3)
        )

        # 마지막 단계: 공급업체 컴플라이언스 확인
        final_messages = updated_messages + transportation_result["messages"]
        supplier_result = await workflow.execute_activity(
            "supplier_activity",
            {"operation": operation, "messages": final_messages},
            start_to_close_timeout=timedelta(seconds=30),
            retry_policy=RetryPolicy(maximum_attempts=3)
        )

        # 단계별 결과를 모아서 반환
        return {
            "inventory": inventory_result,
            "transportation": transportation_result,
            "supplier": supplier_result
        }

* 워크플로 엔진은 더 높은 추상화를 제공하여 조율 로직을 통신 메커니즘과 분리  
    -> 멱등성, 복구 가능성, 영속 상태를 보장하는 데 도움을 주며 에이전트가 실패하거나 멈추거나 변화하는 환경에 대응해야 할 때 필수적인 특성

## 상태와 영속성 관리

* 멀티 에이전트 시스템은 여러 번의 실행, 워크플로, 시스템 재시작에 걸쳐 존재하는 공유 상태, 에이전트 메모리, 작업별 메타데이터도 함께 관리해야 함  
    -> 데이터 영속성, 일관성, 접근 패턴 측면에서 상당한 복잡성이 생기고 시스템이 확장될수록 문제는 심화

* 기존의 솔루션은 PostgreSQL, Redis 같은 상태 저장 DB나 벡터 스토어에 의존해 작업 결과, 상호작용 로그, 애이전트 메모리를 영속화  
    -> 세밀한 제어를 제공, 에이전트별 요구사항에 맞춰 조율할 수 있으나 스키마 설계, 읽기/쓰기 일관성, 캐싱, 복구 로직을 개발자가 직접 관리해야 함  
    -> 엔지니어링 오버헤드 + 미묘한 버그 발생 가능

* 비구조화되었거나 대규모인 출력의 경우 Amazon S3나 Azure Blob Storage 같은 오브젝트 스토리지는 높은 가용성을 가진 내구성 있는 저비용 저장소를 제공  
    -> 변경되지 않는 아티팩트를 보관하는 데는 이상적, 접근 지연시간이 길어질 수 있고 아티팩트를 에이전트 작업이나 상태와 다시 연결하기 위한 별도의 인덱싱 또는 추적 시스템이 필요한 트레이드오프가 존재 

* 영속 저장소

| 접근 방식 | 장점 | 단점 | 적합한 용도 |
|-----|-----|-----|-----|
| 상태 저장 DB | 유연하고 질의 가능하며 비용 효율적 | 수동 관리 필요, 일관성 문제 가능성 | 맞춤형, 질의가 많은 시스템 |
| 벡터 스토어 | 시맨틱 검색, 확장 가능한 임베딩 | 더 높은 비용, 특수한 설정 필요 | 지식 집약적인 에이전트 |
| 오브젝트 스토리지 | 저렴하고 대용량 데이터에 대한 영속성이 높음 | 느린 접근, 기본 제공 인덱싱없음 | 아카이브용 출력물 |
| 상태 저장 오케스트레이션 프레임워크 | 자동 복구, 보일러플레이트 코드가 적음 | 프레임워크 종속 | 복원력이 높고 장시간 실행되는 워크플로 |

* 올바른 선택은 필요한 메모리와 조율 방식의 특성에 따라 달라짐
    * 에피소드 메모리
    * 시맨틱 메모리
    * 워크플로 내구성

* 궁극적으로 영속성 설계는 개발자 노력, 성능, 영속성, 유연성 사이의 트레이드오프를 반영